In [2]:
from __future__ import annotations

import json
import math
import random
import shutil
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import ImageFile
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import (
    EfficientNet_B0_Weights,
    MobileNet_V2_Weights,
    VGG16_Weights,
    efficientnet_b0,
    mobilenet_v2,
    vgg16,
)
from tqdm import tqdm


# =============================================================================
# CONFIGURATION - change only this section when needed
# =============================================================================

PROJECT_ROOT = Path(r"C:\Users\purti.balani\Downloads\CAPSTONE")
DATA_DIR = PROJECT_ROOT / "processed_data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "transfer_learning"
MODEL_DIR = PROJECT_ROOT / "models"

MODELS_TO_TRAIN = ["vgg16", "mobilenet_v2", "efficientnet_b0"]

IMAGE_SIZE = 224
BATCH_SIZE = 16          # reduce to 8 if VGG16 gives CUDA out-of-memory
NUM_WORKERS = 0          # keep 0 on Windows; increase to 2/4 only after it works
SEED = 42

EPOCHS_FROZEN = 3        # train the new classification head
EPOCHS_FINE_TUNE = 1     # unfreeze the last convolutional blocks
LR_HEAD = 1e-3
LR_FINE_TUNE = 1e-5
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5

# Medical images can be sensitive to mirroring. Set True only if the team agrees.
USE_HORIZONTAL_FLIP = False

# Leave None when class folder contains "tumor" or "cancer", for example "Brain Tumor".
# Set to the exact folder name only if automatic tumor-class detection fails.
TUMOR_CLASS_NAME_OVERRIDE = None

# ImageNet normalization used by torchvision pretrained classification weights.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

ImageFile.LOAD_TRUNCATED_IMAGES = True


# =============================================================================
# REPRODUCIBILITY AND SETUP
# =============================================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def make_output_dirs() -> None:
    for directory in [OUTPUT_DIR, MODEL_DIR, OUTPUT_DIR / "plots", OUTPUT_DIR / "reports"]:
        directory.mkdir(parents=True, exist_ok=True)


def validate_data_directory(data_dir: Path) -> None:
    expected_splits = ["train", "val", "test"]
    missing = [split for split in expected_splits if not (data_dir / split).exists()]

    if missing:
        raise FileNotFoundError(
            f"Missing split folder(s): {missing}\n"
            f"Expected structure: {data_dir}\\train, {data_dir}\\val, {data_dir}\\test"
        )

    for split in expected_splits:
        class_dirs = [p for p in (data_dir / split).iterdir() if p.is_dir()]
        if len(class_dirs) < 2:
            raise FileNotFoundError(
                f"{data_dir / split} must contain at least two class folders, "
                "for example Brain Tumor and Healthy."
            )


# =============================================================================
# DATA PIPELINE
# =============================================================================

def build_transforms() -> tuple[transforms.Compose, transforms.Compose]:
    train_ops: list[Any] = [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomRotation(degrees=10),
        transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.08, contrast=0.08),
    ]

    if USE_HORIZONTAL_FLIP:
        train_ops.append(transforms.RandomHorizontalFlip(p=0.5))

    train_ops.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    eval_ops = [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]

    return transforms.Compose(train_ops), transforms.Compose(eval_ops)


def create_datasets_and_loaders(
    device: torch.device,
) -> tuple[dict[str, datasets.ImageFolder], dict[str, DataLoader]]:
    validate_data_directory(DATA_DIR)
    train_tfms, eval_tfms = build_transforms()

    dataset_map = {
        "train": datasets.ImageFolder(DATA_DIR / "train", transform=train_tfms),
        "val": datasets.ImageFolder(DATA_DIR / "val", transform=eval_tfms),
        "test": datasets.ImageFolder(DATA_DIR / "test", transform=eval_tfms),
    }

    train_mapping = dataset_map["train"].class_to_idx

    for split in ["val", "test"]:
        if dataset_map[split].class_to_idx != train_mapping:
            raise ValueError(
                f"Class folders in {split} do not match train.\n"
                f"train mapping: {train_mapping}\n"
                f"{split} mapping: {dataset_map[split].class_to_idx}"
            )

    loader_kwargs = {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": device.type == "cuda",
    }

    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = True

    loader_map = {
        "train": DataLoader(dataset_map["train"], shuffle=True, **loader_kwargs),
        "val": DataLoader(dataset_map["val"], shuffle=False, **loader_kwargs),
        "test": DataLoader(dataset_map["test"], shuffle=False, **loader_kwargs),
    }

    return dataset_map, loader_map


def infer_tumor_class_idx(class_to_idx: dict[str, int]) -> int:
    if TUMOR_CLASS_NAME_OVERRIDE is not None:
        if TUMOR_CLASS_NAME_OVERRIDE not in class_to_idx:
            raise ValueError(
                f"TUMOR_CLASS_NAME_OVERRIDE={TUMOR_CLASS_NAME_OVERRIDE!r} not found. "
                f"Available classes: {list(class_to_idx)}"
            )
        return class_to_idx[TUMOR_CLASS_NAME_OVERRIDE]

    candidates: list[tuple[str, int]] = []

    for class_name, idx in class_to_idx.items():
        normalized = class_name.lower().replace("_", " ").replace("-", " ")
        if "tumor" in normalized or "tumour" in normalized or "cancer" in normalized:
            candidates.append((class_name, idx))

    if len(candidates) != 1:
        raise ValueError(
            "Could not automatically detect the tumor/cancer class folder. "
            f"Detected candidates: {candidates}. Available classes: {class_to_idx}. "
            "Set TUMOR_CLASS_NAME_OVERRIDE at the top of the script."
        )

    return candidates[0][1]


def dataset_class_counts(dataset: datasets.ImageFolder) -> np.ndarray:
    labels = [label for _, label in dataset.samples]
    return np.bincount(labels, minlength=len(dataset.classes))


def class_weights_from_dataset(dataset: datasets.ImageFolder, device: torch.device) -> torch.Tensor:
    counts = dataset_class_counts(dataset).astype(np.float32)
    counts = np.maximum(counts, 1.0)

    # Balanced weight formula: total_samples / (num_classes * class_count)
    weights = counts.sum() / (len(counts) * counts)

    return torch.tensor(weights, dtype=torch.float32, device=device)


def print_dataset_summary(dataset_map: dict[str, datasets.ImageFolder], tumor_idx: int) -> None:
    class_names = dataset_map["train"].classes

    print("\nDataset summary")
    print("=" * 72)
    print(f"Data directory: {DATA_DIR}")
    print(f"Class mapping: {dataset_map['train'].class_to_idx}")
    print(f"Tumor positive class: {class_names[tumor_idx]!r} -> index {tumor_idx}")

    for split, dataset in dataset_map.items():
        counts = dataset_class_counts(dataset)
        total = int(counts.sum())
        parts = ", ".join(f"{class_names[i]}={int(counts[i])}" for i in range(len(class_names)))
        print(f"{split:>5}: total={total:<5} | {parts}")

    print("=" * 72)


# =============================================================================
# MODEL BUILDING
# =============================================================================

def freeze_all(model: nn.Module) -> None:
    for param in model.parameters():
        param.requires_grad = False


def build_model(model_name: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    model_name = model_name.lower()

    if model_name == "vgg16":
        weights = VGG16_Weights.DEFAULT if pretrained else None
        model = vgg16(weights=weights)
        freeze_all(model)

        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)

        return model

    if model_name == "mobilenet_v2":
        weights = MobileNet_V2_Weights.DEFAULT if pretrained else None
        model = mobilenet_v2(weights=weights)
        freeze_all(model)

        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

        return model

    if model_name == "efficientnet_b0":
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = efficientnet_b0(weights=weights)
        freeze_all(model)

        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

        return model

    raise ValueError(f"Unknown model_name={model_name!r}. Use one of: {MODELS_TO_TRAIN}")


def load_checkpoint_for_inference(
    checkpoint_path: Path,
    device: torch.device,
) -> tuple[nn.Module, dict[str, Any]]:
    """
    Optional helper for later Streamlit/prediction.py integration.
    This rebuilds the model without downloading ImageNet weights.
    """
    checkpoint = safe_torch_load(checkpoint_path, map_location=device)
    num_classes = len(checkpoint["class_to_idx"])

    model = build_model(
        checkpoint["model_name"],
        num_classes=num_classes,
        pretrained=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    return model, checkpoint


def unfreeze_for_fine_tuning(model_name: str, model: nn.Module) -> None:
    model_name = model_name.lower()

    if model_name == "vgg16":
        # Last VGG feature layers roughly correspond to the final convolution block.
        for param in model.features[-8:].parameters():
            param.requires_grad = True

        for param in model.classifier.parameters():
            param.requires_grad = True

        return

    if model_name == "mobilenet_v2":
        for param in model.features[-4:].parameters():
            param.requires_grad = True

        for param in model.classifier.parameters():
            param.requires_grad = True

        return

    if model_name == "efficientnet_b0":
        for param in model.features[-3:].parameters():
            param.requires_grad = True

        for param in model.classifier.parameters():
            param.requires_grad = True

        return

    raise ValueError(f"Unknown model_name={model_name!r}")


def parameter_summary(model: nn.Module) -> tuple[int, int]:
    total = sum(param.numel() for param in model.parameters())
    trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
    return total, trainable


def make_optimizer(model: nn.Module, learning_rate: float) -> optim.Optimizer:
    trainable_params = [param for param in model.parameters() if param.requires_grad]

    if not trainable_params:
        raise RuntimeError("No trainable parameters found. Check model freezing logic.")

    return optim.AdamW(trainable_params, lr=learning_rate, weight_decay=WEIGHT_DECAY)


# =============================================================================
# METRICS
# =============================================================================

def predictions_from_threshold(
    y_true: np.ndarray,
    probs: np.ndarray,
    tumor_idx: int,
    threshold: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if probs.shape[1] != 2:
        raise ValueError("This helper is written for binary classification only.")

    other_idx = 1 - tumor_idx

    y_true_pos = (y_true == tumor_idx).astype(int)
    tumor_probs = probs[:, tumor_idx]
    y_pred_pos = (tumor_probs >= threshold).astype(int)
    y_pred = np.where(y_pred_pos == 1, tumor_idx, other_idx).astype(int)

    return y_true_pos, y_pred_pos, y_pred


def compute_metrics(
    y_true: np.ndarray,
    probs: np.ndarray,
    tumor_idx: int,
    threshold: float = 0.50,
) -> dict[str, float]:
    y_true_pos, y_pred_pos, y_pred = predictions_from_threshold(
        y_true=y_true,
        probs=probs,
        tumor_idx=tumor_idx,
        threshold=threshold,
    )

    tumor_probs = probs[:, tumor_idx]

    try:
        auc = float(roc_auc_score(y_true_pos, tumor_probs))
    except ValueError:
        auc = float("nan")

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_tumor": float(precision_score(y_true_pos, y_pred_pos, zero_division=0)),
        "recall_tumor": float(recall_score(y_true_pos, y_pred_pos, zero_division=0)),
        "f1_tumor": float(f1_score(y_true_pos, y_pred_pos, zero_division=0)),
        "roc_auc_tumor": auc,
        "threshold": float(threshold),
    }


def selection_score(metrics: dict[str, float]) -> float:
    """
    Validation selection score.

    The capstone plan says tumor-class recall is the key metric.
    Therefore, recall gets the highest weight here.
    """
    return (
        0.50 * metrics["recall_tumor"]
        + 0.35 * metrics["f1_tumor"]
        + 0.15 * metrics["accuracy"]
    )


def find_best_threshold(
    y_true: np.ndarray,
    probs: np.ndarray,
    tumor_idx: int,
) -> tuple[float, dict[str, float], float]:
    """
    Tune the tumor probability threshold on validation data only.

    Lower threshold can increase tumor recall.
    Higher threshold can increase precision.
    This script picks the threshold with best recall-weighted validation score.
    """
    best_threshold = 0.50
    best_metrics = compute_metrics(y_true, probs, tumor_idx, threshold=best_threshold)
    best_score = selection_score(best_metrics)

    for threshold in np.round(np.arange(0.05, 0.96, 0.01), 2):
        metrics = compute_metrics(y_true, probs, tumor_idx, threshold=float(threshold))
        score = selection_score(metrics)

        if score > best_score:
            best_threshold = float(threshold)
            best_metrics = metrics
            best_score = score

    return best_threshold, best_metrics, best_score


# =============================================================================
# TRAINING AND EVALUATION
# =============================================================================

def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    epoch_label: str,
) -> tuple[float, float]:
    model.train()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    progress = tqdm(data_loader, desc=epoch_label, leave=False)

    for images, labels in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += batch_size

        progress.set_postfix(
            loss=f"{running_loss / max(total_samples, 1):.4f}",
            acc=f"{running_correct / max(total_samples, 1):.4f}",
        )

    return running_loss / total_samples, running_correct / total_samples


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    data_loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    tumor_idx: int,
    threshold: float = 0.50,
) -> tuple[float, dict[str, float], np.ndarray, np.ndarray]:
    model.eval()

    running_loss = 0.0
    total_samples = 0
    all_labels: list[np.ndarray] = []
    all_probs: list[np.ndarray] = []

    for images, labels in tqdm(data_loader, desc="Evaluating", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)
        probs = torch.softmax(logits, dim=1)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

        all_labels.append(labels.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())

    y_true = np.concatenate(all_labels)
    probs = np.concatenate(all_probs)

    metrics = compute_metrics(y_true, probs, tumor_idx, threshold=threshold)

    return running_loss / total_samples, metrics, y_true, probs


# =============================================================================
# SAVING, PLOTTING, REPORTING
# =============================================================================

def clean_for_json(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): clean_for_json(v) for k, v in value.items()}

    if isinstance(value, list):
        return [clean_for_json(v) for v in value]

    if isinstance(value, tuple):
        return [clean_for_json(v) for v in value]

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return None if math.isnan(float(value)) else float(value)

    if isinstance(value, float):
        return None if math.isnan(value) else value

    return value


def save_json(data: Any, path: Path) -> None:
    with path.open("w", encoding="utf-8") as file:
        json.dump(clean_for_json(data), file, indent=2)


def safe_torch_load(path: Path, map_location: torch.device | str) -> dict[str, Any]:
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def save_checkpoint(
    path: Path,
    model: nn.Module,
    model_name: str,
    class_to_idx: dict[str, int],
    tumor_idx: int,
    epoch: int,
    phase: str,
    threshold: float,
    metrics: dict[str, Any],
) -> None:
    checkpoint = {
        "model_name": model_name,
        "model_state_dict": model.state_dict(),
        "class_to_idx": class_to_idx,
        "idx_to_class": {idx: name for name, idx in class_to_idx.items()},
        "tumor_idx": tumor_idx,
        "image_size": IMAGE_SIZE,
        "imagenet_mean": IMAGENET_MEAN,
        "imagenet_std": IMAGENET_STD,
        "threshold": threshold,
        "epoch": epoch,
        "phase": phase,
        "metrics": clean_for_json(metrics),
    }

    torch.save(checkpoint, path)


def plot_history(history: list[dict[str, Any]], path: Path, model_name: str) -> None:
    if not history:
        return

    history_df = pd.DataFrame(history)
    history_df.to_csv(path.with_suffix(".csv"), index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    plt.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path.parent / f"{model_name}_loss.png", dpi=160)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["train_acc"], label="train_accuracy")
    plt.plot(history_df["epoch"], history_df["val_accuracy"], label="val_accuracy")
    plt.plot(history_df["epoch"], history_df["val_recall_tumor"], label="val_recall_tumor")
    plt.plot(history_df["epoch"], history_df["val_f1_tumor"], label="val_f1_tumor")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title(f"{model_name} validation metrics")
    plt.legend()
    plt.tight_layout()
    plt.savefig(path.parent / f"{model_name}_metrics.png", dpi=160)
    plt.close()


def save_confusion_matrix_plot(
    y_true: np.ndarray,
    probs: np.ndarray,
    tumor_idx: int,
    threshold: float,
    class_names: list[str],
    path: Path,
    title: str,
) -> np.ndarray:
    _, _, y_pred = predictions_from_threshold(
        y_true=y_true,
        probs=probs,
        tumor_idx=tumor_idx,
        threshold=threshold,
    )

    labels = list(range(len(class_names)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    plt.xticks(labels, class_names, rotation=30, ha="right")
    plt.yticks(labels, class_names)
    plt.xlabel("Predicted label")
    plt.ylabel("True label")

    for row in range(cm.shape[0]):
        for col in range(cm.shape[1]):
            plt.text(col, row, str(cm[row, col]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()

    return cm


# =============================================================================
# MAIN MODEL TRAINING LOOP
# =============================================================================

def train_transfer_model(
    model_name: str,
    dataset_map: dict[str, datasets.ImageFolder],
    loader_map: dict[str, DataLoader],
    device: torch.device,
    tumor_idx: int,
) -> dict[str, Any]:
    print(f"\n\nTraining {model_name}")
    print("=" * 72)

    num_classes = len(dataset_map["train"].classes)
    class_to_idx = dataset_map["train"].class_to_idx
    class_names = dataset_map["train"].classes

    model = build_model(model_name, num_classes=num_classes, pretrained=True).to(device)

    total_params, trainable_params = parameter_summary(model)
    print(f"Initial trainable params: {trainable_params:,} / {total_params:,}")

    class_weights = class_weights_from_dataset(dataset_map["train"], device=device)
    print(f"Class weights for loss: {class_weights.detach().cpu().numpy().round(4).tolist()}")

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_checkpoint_path = MODEL_DIR / f"{model_name}_best.pth"
    history: list[dict[str, Any]] = []

    best_score = -1.0
    best_epoch = 0
    bad_epochs = 0
    global_epoch = 0

    phases = [
        ("head_only", EPOCHS_FROZEN, LR_HEAD),
        ("fine_tune", EPOCHS_FINE_TUNE, LR_FINE_TUNE),
    ]

    start_time = time.time()

    for phase_name, phase_epochs, learning_rate in phases:
        if phase_epochs <= 0:
            continue

        if phase_name == "fine_tune":
            unfreeze_for_fine_tuning(model_name, model)
            total_params, trainable_params = parameter_summary(model)
            print(f"Fine-tuning trainable params: {trainable_params:,} / {total_params:,}")
            bad_epochs = 0

        optimizer = make_optimizer(model, learning_rate=learning_rate)

        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.3,
            patience=2,
        )

        for phase_epoch in range(1, phase_epochs + 1):
            global_epoch += 1

            label = f"{model_name} | {phase_name} | epoch {phase_epoch}/{phase_epochs}"

            train_loss, train_acc = train_one_epoch(
                model=model,
                data_loader=loader_map["train"],
                criterion=criterion,
                optimizer=optimizer,
                device=device,
                epoch_label=label,
            )

            val_loss, val_metrics, _, _ = evaluate_model(
                model=model,
                data_loader=loader_map["val"],
                criterion=criterion,
                device=device,
                tumor_idx=tumor_idx,
                threshold=0.50,
            )

            score = selection_score(val_metrics)
            scheduler.step(score)

            row = {
                "model": model_name,
                "epoch": global_epoch,
                "phase": phase_name,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_accuracy": val_metrics["accuracy"],
                "val_precision_tumor": val_metrics["precision_tumor"],
                "val_recall_tumor": val_metrics["recall_tumor"],
                "val_f1_tumor": val_metrics["f1_tumor"],
                "val_roc_auc_tumor": val_metrics["roc_auc_tumor"],
                "val_selection_score": score,
                "lr": optimizer.param_groups[0]["lr"],
            }

            history.append(row)

            print(
                f"Epoch {global_epoch:02d} [{phase_name}] "
                f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
                f"val_loss={val_loss:.4f} val_acc={val_metrics['accuracy']:.4f} "
                f"val_recall_tumor={val_metrics['recall_tumor']:.4f} "
                f"val_f1_tumor={val_metrics['f1_tumor']:.4f} score={score:.4f}"
            )

            if score > best_score + 1e-5:
                best_score = score
                best_epoch = global_epoch
                bad_epochs = 0

                save_checkpoint(
                    path=best_checkpoint_path,
                    model=model,
                    model_name=model_name,
                    class_to_idx=class_to_idx,
                    tumor_idx=tumor_idx,
                    epoch=global_epoch,
                    phase=phase_name,
                    threshold=0.50,
                    metrics={
                        "validation_metrics_at_0_50": val_metrics,
                        "selection_score": score,
                    },
                )

                print(f"Saved improved checkpoint: {best_checkpoint_path}")

            else:
                bad_epochs += 1

            if bad_epochs >= EARLY_STOPPING_PATIENCE:
                print(
                    f"Early stopping {model_name} during {phase_name} "
                    f"after {bad_epochs} non-improving epochs."
                )
                break

    elapsed_minutes = (time.time() - start_time) / 60.0

    plot_history(
        history=history,
        path=OUTPUT_DIR / "plots" / f"{model_name}_history.csv",
        model_name=model_name,
    )

    # Load best checkpoint.
    checkpoint = safe_torch_load(best_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    # Tune threshold on validation set only.
    val_loss, val_metrics_050, y_val, probs_val = evaluate_model(
        model=model,
        data_loader=loader_map["val"],
        criterion=criterion,
        device=device,
        tumor_idx=tumor_idx,
        threshold=0.50,
    )

    best_threshold, val_metrics_tuned, val_score_tuned = find_best_threshold(
        y_true=y_val,
        probs=probs_val,
        tumor_idx=tumor_idx,
    )

    # Final test evaluation using the validation-selected threshold.
    test_loss, test_metrics_tuned, y_test, probs_test = evaluate_model(
        model=model,
        data_loader=loader_map["test"],
        criterion=criterion,
        device=device,
        tumor_idx=tumor_idx,
        threshold=best_threshold,
    )

    cm = save_confusion_matrix_plot(
        y_true=y_test,
        probs=probs_test,
        tumor_idx=tumor_idx,
        threshold=best_threshold,
        class_names=class_names,
        path=OUTPUT_DIR / "plots" / f"{model_name}_test_confusion_matrix.png",
        title=f"{model_name} test confusion matrix",
    )

    _, _, y_test_pred = predictions_from_threshold(
        y_true=y_test,
        probs=probs_test,
        tumor_idx=tumor_idx,
        threshold=best_threshold,
    )

    report_text = classification_report(
        y_test,
        y_test_pred,
        target_names=class_names,
        zero_division=0,
        digits=4,
    )

    report_path = OUTPUT_DIR / "reports" / f"{model_name}_classification_report.txt"
    report_path.write_text(report_text, encoding="utf-8")

    final_metrics = {
        "model": model_name,
        "best_epoch": best_epoch,
        "training_minutes": elapsed_minutes,
        "best_threshold_from_val": best_threshold,

        "val_loss": val_loss,
        "val_accuracy_0_50": val_metrics_050["accuracy"],
        "val_recall_tumor_0_50": val_metrics_050["recall_tumor"],
        "val_f1_tumor_0_50": val_metrics_050["f1_tumor"],

        "val_accuracy_tuned": val_metrics_tuned["accuracy"],
        "val_precision_tumor_tuned": val_metrics_tuned["precision_tumor"],
        "val_recall_tumor_tuned": val_metrics_tuned["recall_tumor"],
        "val_f1_tumor_tuned": val_metrics_tuned["f1_tumor"],
        "val_roc_auc_tumor": val_metrics_tuned["roc_auc_tumor"],
        "val_selection_score_tuned": val_score_tuned,

        "test_loss": test_loss,
        "test_accuracy": test_metrics_tuned["accuracy"],
        "test_precision_tumor": test_metrics_tuned["precision_tumor"],
        "test_recall_tumor": test_metrics_tuned["recall_tumor"],
        "test_f1_tumor": test_metrics_tuned["f1_tumor"],
        "test_roc_auc_tumor": test_metrics_tuned["roc_auc_tumor"],
        "test_selection_score": selection_score(test_metrics_tuned),

        "confusion_matrix": cm.tolist(),
        "checkpoint_path": str(best_checkpoint_path),
        "classification_report_path": str(report_path),
    }

    # Store tuned threshold and final metrics inside checkpoint for deployment.
    checkpoint.update({
        "threshold": best_threshold,
        "final_metrics": clean_for_json(final_metrics),
    })

    torch.save(checkpoint, best_checkpoint_path)

    save_json(
        final_metrics,
        OUTPUT_DIR / "reports" / f"{model_name}_final_metrics.json",
    )

    print(f"\nFinal test metrics for {model_name} using threshold={best_threshold:.2f}")
    print(report_text)

    return final_metrics


# =============================================================================
# ENTRY POINT
# =============================================================================

def main() -> None:
    set_seed(SEED)
    make_output_dirs()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Using device: {device}")

    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")

    dataset_map, loader_map = create_datasets_and_loaders(device)
    tumor_idx = infer_tumor_class_idx(dataset_map["train"].class_to_idx)

    print_dataset_summary(dataset_map, tumor_idx)

    all_results: list[dict[str, Any]] = []

    for model_name in MODELS_TO_TRAIN:
        result = train_transfer_model(
            model_name=model_name,
            dataset_map=dataset_map,
            loader_map=loader_map,
            device=device,
            tumor_idx=tumor_idx,
        )

        all_results.append(result)

        pd.DataFrame(all_results).to_csv(
            OUTPUT_DIR / "model_comparison_partial.csv",
            index=False,
        )

    comparison_df = pd.DataFrame(all_results)

    comparison_df = comparison_df.sort_values(
        by=[
            "val_selection_score_tuned",
            "test_recall_tumor",
            "test_f1_tumor",
            "test_accuracy",
        ],
        ascending=False,
    )

    comparison_path = OUTPUT_DIR / "model_comparison.csv"
    comparison_df.to_csv(comparison_path, index=False)

    save_json(all_results, OUTPUT_DIR / "model_comparison.json")

    best_model_name = str(comparison_df.iloc[0]["model"])
    source_checkpoint = MODEL_DIR / f"{best_model_name}_best.pth"
    best_checkpoint = MODEL_DIR / "best_transfer_model.pth"

    shutil.copy2(source_checkpoint, best_checkpoint)

    print("\nModel comparison")
    print("=" * 72)

    display_cols = [
        "model",
        "best_threshold_from_val",
        "val_recall_tumor_tuned",
        "val_f1_tumor_tuned",
        "test_accuracy",
        "test_precision_tumor",
        "test_recall_tumor",
        "test_f1_tumor",
        "test_roc_auc_tumor",
    ]

    print(comparison_df[display_cols].to_string(index=False))
    print("=" * 72)

    print(f"Best transfer model selected by validation score: {best_model_name}")
    print(f"Saved comparison CSV: {comparison_path}")
    print(f"Saved best transfer model: {best_checkpoint}")
    print("Use tumor recall, F1-score, confusion matrix, and ROC-AUC in your slides/report.")


if __name__ == "__main__":
    # Required on Windows when DataLoader workers are increased above 0.
    import multiprocessing as mp

    mp.freeze_support()
    main()

Using device: cpu

Dataset summary
Data directory: C:\Users\purti.balani\Downloads\CAPSTONE\processed_data
Class mapping: {'Brain Tumor': 0, 'Healthy': 1}
Tumor positive class: 'Brain Tumor' -> index 0
train: total=3159  | Brain Tumor=1670, Healthy=1489
  val: total=677   | Brain Tumor=360, Healthy=317
 test: total=678   | Brain Tumor=397, Healthy=281


Training vgg16
Initial trainable params: 8,194 / 134,268,738
Class weights for loss: [0.9458000063896179, 1.0607999563217163]


Epoch 01 [head_only] train_loss=0.4814 train_acc=0.7683 val_loss=0.3325 val_acc=0.8626 val_recall_tumor=0.8472 val_f1_tumor=0.8677 score=0.8567
Saved improved checkpoint: C:\Users\purti.balani\Downloads\CAPSTONE\models\vgg16_best.pth


Epoch 02 [head_only] train_loss=0.4031 train_acc=0.8161 val_loss=0.3275 val_acc=0.8626 val_recall_tumor=0.8000 val_f1_tumor=0.8610 score=0.8307


Epoch 03 [head_only] train_loss=0.3918 train_acc=0.8319 val_loss=0.3043 val_acc=0.8671 val_recall_tumor=0.9167 val_f1_tumor=0.8800 score=0.8964
Saved improved checkpoint: C:\Users\purti.balani\Downloads\CAPSTONE\models\vgg16_best.pth
Fine-tuning trainable params: 126,633,474 / 134,268,738


Epoch 04 [fine_tune] train_loss=0.2707 train_acc=0.8870 val_loss=0.1990 val_acc=0.9276 val_recall_tumor=0.8972 val_f1_tumor=0.9295 score=0.9131
Saved improved checkpoint: C:\Users\purti.balani\Downloads\CAPSTONE\models\vgg16_best.pth


...

Epoch 01 [head_only] train_loss=0.4412 train_acc=0.8003 val_loss=0.3577 val_acc=0.8390 val_recall_tumor=0.7528 val_f1_tumor=0.8326 score=0.7936
Saved improved checkpoint: C:\Users\purti.balani\Downloads\CAPSTONE\models\efficientnet_b0_best.pth


Epoch 02 [head_only] train_loss=0.3536 train_acc=0.8424 val_loss=0.3458 val_acc=0.8360 val_recall_tumor=0.7278 val_f1_tumor=0.8252 score=0.7781


Epoch 03 [head_only] train_loss=0.3269 train_acc=0.8594 val_loss=0.3074 val_acc=0.8626 val_recall_tumor=0.7778 val_f1_tumor=0.8576 score=0.8184
Saved improved checkpoint: C:\Users\purti.balani\Downloads\CAPSTONE\models\efficientnet_b0_best.pth
Fine-tuning trainable params: 3,158,302 / 4,010,110


Epoch 04 [fine_tune] train_loss=0.2907 train_acc=0.8778 val_loss=0.2234 val_acc=0.9099 val_recall_tumor=0.8861 val_f1_tumor=0.9127 score=0.8990
Saved improved checkpoint: C:\Users\purti.balani\Downloads\CAPSTONE\models\efficientnet_b0_best.pth



Final test metrics for efficientnet_b0 using threshold=0.21
              precision    recall  f1-score   support

 Brain Tumor     0.8879    0.9773    0.9305       397
     Healthy     0.9627    0.8256    0.8889       281

    accuracy                         0.9145       678
   macro avg     0.9253    0.9015    0.9097       678
weighted avg     0.9189    0.9145    0.9132       678


Model comparison
          model  best_threshold_from_val  val_recall_tumor_tuned  val_f1_tumor_tuned  test_accuracy  test_precision_tumor  test_recall_tumor  test_f1_tumor  test_roc_auc_tumor
          vgg16                     0.18                0.963889            0.931544       0.915929              0.904762           0.957179       0.930233            0.975681
efficientnet_b0                     0.21                0.975000            0.921260       0.914454              0.887872           0.977330       0.930456            0.976102
   mobilenet_v2                     0.37                0.972222  